In [1]:
import pandas as pd

strong_df = pd.read_csv("data/strong_export.csv")
hevy_df = pd.read_csv("data/hevy_export.csv")

strong_df['date'] = pd.to_datetime(strong_df['Date'], format='%Y-%m-%d %H:%M:%S')
hevy_df['date'] = pd.to_datetime(hevy_df['start_time'], format='%d %b %Y, %H:%M')

In [2]:
strong_df['Exercise Name'] = strong_df['Exercise Name'].replace('Squat (Band)', 'Squat (Barbell)')

HEVY_TO_CANONICAL = {
    'Chest Fly (Machine)': 'Pec Deck (Machine)',
    'Rear Delt Reverse Fly (Machine)': 'Reverse Fly (Machine)',
    'Triceps Pushdown': 'Triceps Pushdown (Cable - Straight Bar)',
    'Seated Calf Raise': 'Seated Calf Raise (Machine)',
    'Shoulder Press (Dumbbell)': 'Seated Overhead Press (Dumbbell)',
    'Overhead Triceps Extension (Cable)': 'Triceps Extension (Cable)',
    'Hanging Leg Raise': 'Leg Raise Parallel Bars',
}
hevy_df['exercise_title'] = hevy_df['exercise_title'].replace(HEVY_TO_CANONICAL)

In [3]:
strong_clean = strong_df.rename(columns={
    'Exercise Name': 'exercise',
    'Weight': 'weight',
    'Reps': 'reps',
    'Set Order': 'set_order',
})[['date', 'exercise', 'weight', 'reps', 'set_order']]
strong_clean['source'] = 'strong'

hevy_clean = hevy_df.rename(columns={
    'exercise_title': 'exercise',
    'weight_lbs': 'weight',
    'reps': 'reps',
    'set_index': 'set_order',
})[['date', 'exercise', 'weight', 'reps', 'set_order']]
hevy_clean['source'] = 'hevy'

In [4]:
combined = pd.concat([strong_clean, hevy_clean], ignore_index=True)
combined = combined.sort_values('date').reset_index(drop=True)
combined

,date,exercise,weight,reps,set_order,source
0,2025-09-19 10:30:11,Squat (Barbell),135.0,8.0,1,strong
1,2025-09-19 10:30:11,Lat Pulldown (Cable),130.0,7.0,2,strong
2,2025-09-19 10:30:11,Bench Press (Barbell),95.0,13.0,2,strong
3,2025-09-19 10:30:11,Bench Press (Barbell),95.0,14.0,1,strong
4,2025-09-19 10:30:11,Seated Leg Curl (Machine),130.0,10.0,3,strong
...,...,...,...,...,...,...
1223,2026-07-04 13:24:00,Seated Row (Machine),145.0,6.0,0,hevy
1224,2026-07-04 13:24:00,Pull Up,NaN,5.0,1,hevy
1225,2026-07-04 13:24:00,Pull Up,NaN,5.0,0,hevy
1226,2026-07-04 13:24:00,Preacher Curl (Machine),110.0,6.0,1,hevy


In [5]:
BODYWEIGHT_EXERCISES = ['Pull Up', 'Leg Raise Parallel Bars']

modeling_df = combined[~combined['exercise'].isin(BODYWEIGHT_EXERCISES)].copy()

print("combined rows:", len(combined))
print("modeling_df rows:", len(modeling_df))
print("Any bodyweight exercises left?", modeling_df['exercise'].isin(BODYWEIGHT_EXERCISES).any())

combined rows: 1228
modeling_df rows: 1209
Any bodyweight exercises left? False


In [6]:
sorted_df = modeling_df.sort_values(['date', 'exercise', 'weight', 'reps'])
e1rm_df = sorted_df.drop_duplicates(subset=['date', 'exercise'], keep='last').copy()

print("modeling_df rows:", len(modeling_df))
print("e1rm_df rows:", len(e1rm_df))
e1rm_df.head(10)

modeling_df rows: 1209
e1rm_df rows: 434


,date,exercise,weight,reps,set_order,source
3,2025-09-19 10:30:11,Bench Press (Barbell),95.0,14.0,1,strong
6,2025-09-19 10:30:11,Lat Pulldown (Cable),130.0,9.0,1,strong
9,2025-09-19 10:30:11,Seated Calf Raise (Machine),70.0,11.0,1,strong
5,2025-09-19 10:30:11,Seated Leg Curl (Machine),130.0,12.0,2,strong
11,2025-09-19 10:30:11,Squat (Barbell),145.0,6.0,2,strong
31,2025-09-22 14:15:46,Bench Press (Barbell),125.0,5.0,2,strong
27,2025-09-22 14:15:46,Incline Bench Press (Dumbbell),40.0,6.0,3,strong
19,2025-09-22 14:15:46,Lateral Raise (Dumbbell),20.0,9.0,1,strong
13,2025-09-22 14:15:46,Pec Deck (Machine),135.0,8.0,1,strong
24,2025-09-22 14:15:46,Seated Overhead Press (Dumbbell),35.0,7.0,3,strong


In [7]:
e1rm_df['e1rm'] = (e1rm_df['weight'] * (1 + e1rm_df['reps'] / 30)).round(2)
e1rm_df.head(10)

,date,exercise,weight,reps,set_order,source,e1rm
3,2025-09-19 10:30:11,Bench Press (Barbell),95.0,14.0,1,strong,139.33
6,2025-09-19 10:30:11,Lat Pulldown (Cable),130.0,9.0,1,strong,169.00
9,2025-09-19 10:30:11,Seated Calf Raise (Machine),70.0,11.0,1,strong,95.67
5,2025-09-19 10:30:11,Seated Leg Curl (Machine),130.0,12.0,2,strong,182.00
11,2025-09-19 10:30:11,Squat (Barbell),145.0,6.0,2,strong,174.00
31,2025-09-22 14:15:46,Bench Press (Barbell),125.0,5.0,2,strong,145.83
27,2025-09-22 14:15:46,Incline Bench Press (Dumbbell),40.0,6.0,3,strong,48.00
19,2025-09-22 14:15:46,Lateral Raise (Dumbbell),20.0,9.0,1,strong,26.00
13,2025-09-22 14:15:46,Pec Deck (Machine),135.0,8.0,1,strong,171.00
24,2025-09-22 14:15:46,Seated Overhead Press (Dumbbell),35.0,7.0,3,strong,43.17


In [8]:
rolling = (
    e1rm_df.groupby('exercise')
    .rolling('28D', on='date')['e1rm']
    .mean()
    .reset_index()
    .rename(columns={'e1rm': 'rolling_e1rm'})
)

e1rm_df = e1rm_df.merge(rolling[['exercise', 'date', 'rolling_e1rm']], on=['exercise', 'date'], how='left')

e1rm_df.head(10)

,date,exercise,weight,reps,set_order,source,e1rm,rolling_e1rm
0,2025-09-19 10:30:11,Bench Press (Barbell),95.0,14.0,1,strong,139.33,139.33
1,2025-09-19 10:30:11,Lat Pulldown (Cable),130.0,9.0,1,strong,169.00,169.00
2,2025-09-19 10:30:11,Seated Calf Raise (Machine),70.0,11.0,1,strong,95.67,95.67
3,2025-09-19 10:30:11,Seated Leg Curl (Machine),130.0,12.0,2,strong,182.00,182.00
4,2025-09-19 10:30:11,Squat (Barbell),145.0,6.0,2,strong,174.00,174.00
5,2025-09-22 14:15:46,Bench Press (Barbell),125.0,5.0,2,strong,145.83,142.58
6,2025-09-22 14:15:46,Incline Bench Press (Dumbbell),40.0,6.0,3,strong,48.00,48.00
7,2025-09-22 14:15:46,Lateral Raise (Dumbbell),20.0,9.0,1,strong,26.00,26.00
8,2025-09-22 14:15:46,Pec Deck (Machine),135.0,8.0,1,strong,171.00,171.00
9,2025-09-22 14:15:46,Seated Overhead Press (Dumbbell),35.0,7.0,3,strong,43.17,43.17


In [9]:
e1rm_df['intensity_trend'] = (e1rm_df['weight'] / e1rm_df['rolling_e1rm']).round(4)
e1rm_df.head(10)

,date,exercise,weight,reps,set_order,source,e1rm,rolling_e1rm,intensity_trend
0,2025-09-19 10:30:11,Bench Press (Barbell),95.0,14.0,1,strong,139.33,139.33,0.6818
1,2025-09-19 10:30:11,Lat Pulldown (Cable),130.0,9.0,1,strong,169.00,169.00,0.7692
2,2025-09-19 10:30:11,Seated Calf Raise (Machine),70.0,11.0,1,strong,95.67,95.67,0.7317
3,2025-09-19 10:30:11,Seated Leg Curl (Machine),130.0,12.0,2,strong,182.00,182.00,0.7143
4,2025-09-19 10:30:11,Squat (Barbell),145.0,6.0,2,strong,174.00,174.00,0.8333
5,2025-09-22 14:15:46,Bench Press (Barbell),125.0,5.0,2,strong,145.83,142.58,0.8767
6,2025-09-22 14:15:46,Incline Bench Press (Dumbbell),40.0,6.0,3,strong,48.00,48.00,0.8333
7,2025-09-22 14:15:46,Lateral Raise (Dumbbell),20.0,9.0,1,strong,26.00,26.00,0.7692
8,2025-09-22 14:15:46,Pec Deck (Machine),135.0,8.0,1,strong,171.00,171.00,0.7895
9,2025-09-22 14:15:46,Seated Overhead Press (Dumbbell),35.0,7.0,3,strong,43.17,43.17,0.8107
